# Chapter 8: Domain-Specific Neural Architectures

```{admonition} Learning Objectives
:class: tip
- Understand convolutional neural networks (CNNs)
- Master convolution, pooling, and padding operations
- Apply CNNs to image data
- Understand recurrent neural networks (RNNs)
- Implement LSTM and GRU architectures
- Apply backpropagation through time (BPTT)
- Build sequence models for text and time-series
```

```{epigraph}
The secret to the success of any neural architecture lies in designing the architecture in a way that is sensitive to the understanding of the domain at hand.

-- Charu Aggarwal
```

## 8.1 Introduction

While feedforward neural networks (Chapter 7) are general-purpose function approximators, **domain-specific architectures** exploit the structure inherent in specific data types to achieve superior performance.

### Structured Data Types

**Grid-Structured Data** (Images):
- 2D spatial relationships
- Adjacent pixels correlated
- Translation invariance
- **Solution**: Convolutional Neural Networks (CNNs)

**Sequential Data** (Text, Audio, Time-Series):
- Temporal dependencies
- Variable length sequences
- Long-range relationships
- **Solution**: Recurrent Neural Networks (RNNs)

### Key Principles

1. **Parameter Sharing**: Same parameters applied to different spatial/temporal locations
2. **Sparse Connectivity**: Each unit connected only to local region
3. **Domain-Aware Regularization**: Architecture design based on domain knowledge
4. **Hierarchical Feature Learning**: Complex features built from simpler ones

### Biological Inspiration

**Hubel & Wiesel's Experiments** (1959):
- Cat visual cortex has cells sensitive to specific regions
- Different cells respond to different orientations
- Hierarchical organization of visual processing
- Led to neocognitron (1980) → LeNet-5 (1998) → modern CNNs

**Success Story**: ImageNet ILSVRC competition
- 2012: AlexNet breakthrough (CNN)
- Top-5 error: 16.4% (vs 26% previous year)
- Sparked deep learning revolution

## 8.2 Convolutional Neural Networks (CNNs)

CNNs are designed for **grid-structured data** with strong spatial dependencies.

### 8.2.1 Basic Structure

**3D Layer Organization**:

Each layer has dimensions: **Height × Width × Depth**

- **Height** ($L$): Spatial dimension
- **Width** ($B$): Spatial dimension
- **Depth** ($d$): Number of feature maps/channels

**Example**: Input image $32 \times 32 \times 3$
- Height/Width: 32 pixels
- Depth: 3 channels (RGB)

**Hidden layers**: Depth increases, spatial dimensions usually decrease

### 8.2.2 Convolution Operation

**Filter/Kernel**: 3D block of parameters
- Spatial dimensions: $F \times F$ (typically 3×3 or 5×5)
- Depth: Same as input layer
- Example: $5 \times 5 \times 3$ filter for RGB input

**Convolution**: Slide filter over input, compute dot products

$$h_{ijp}^{(q+1)} = \sum_{r=1}^{F_q} \sum_{s=1}^{F_q} \sum_{k=1}^{d_q} w_{rsk}^{p,q} h_{i+r-1,j+s-1,k}^{(q)}$$

where:
- $h_{ijk}^{(q)}$: activation at position $(i,j)$, depth $k$, layer $q$
- $w_{rsk}^{p,q}$: weight in filter $p$ at position $(r,s,k)$
- $F_q$: filter spatial size

**Output Dimensions** (without padding/stride):
$$L_{q+1} = L_q - F_q + 1$$
$$B_{q+1} = B_q - F_q + 1$$
$$d_{q+1} = \text{number of filters}$$

**Key Property**: **Translation Equivariance**
- Shifting input shifts output by same amount
- Same feature detected everywhere

### 8.2.3 Example Convolution

**Input**: $7 \times 7 \times 1$ (grayscale)

```
Input:        Filter:      Output:
1 0 1 0 ...   1  0 -1      16 16 26
0 2 0 6 ...   2  0 -2      18 25 14
6 3 4 4 ...   1  0 -1      ...
...
```

**Computation at position (0,0)**:
$$\begin{align}
o_{00} &= 1 \cdot 5 + 0 \cdot 1 + (-1) \cdot 8 \\ 
       &+ 2 \cdot 8 + 0 \cdot 1 + (-2) \cdot 1 \\
       &+ 1 \cdot 1 + 0 \cdot 2 + (-1) \cdot 0 \\
       &= 16
\end{align}$$

## 8.3 Padding

**Problem**: Convolution reduces spatial dimensions

**Solution**: Add zeros around borders

### 8.3.1 Types of Padding

**1. Valid Padding** (No padding):
- Output size: $(L - F + 1) \times (B - F + 1)$
- Border information underrepresented

**2. Half Padding** (Same padding):
- Add $\lfloor F/2 \rfloor$ zeros on each side
- **Output size = Input size**
- Most common choice

For $32 \times 32$ input with $5 \times 5$ filter:
- Pad with 2 zeros: $32 \to 36 \to 32$

**3. Full Padding**:
- Add $(F-1)$ zeros on each side
- Output size: $(L + F - 1) \times (B + F - 1)$
- Used in deconvolution

### 8.3.2 Why Padding Matters

**Without padding**:
- Border pixels contribute to fewer outputs
- Information loss compounds over layers
- Spatial resolution decreases rapidly

**With padding**:
- All positions equally represented
- Maintain spatial dimensions
- Better feature extraction

## 8.4 Strides and Pooling

### 8.4.1 Strides

**Stride** ($S$): Step size when sliding filter

**Output dimensions with stride**:
$$L_{q+1} = \left\lfloor \frac{L_q - F_q}{S_q} \right\rfloor + 1$$

**Common values**:
- $S=1$: Standard, maximum information
- $S=2$: Downsampling, reduces computation
- $S>2$: Rarely used

**Effect**:
- Increases receptive field rapidly
- Reduces spatial dimensions by factor $\approx S^2$
- Less computation, more memory efficient

### 8.4.2 Pooling

**Pooling**: Downsampling operation, typically with $P \times P$ window

**Max Pooling** (most common):
$$h_{ijp}^{(q+1)} = \max_{r,s \in [1,P]} h_{Si+r, Sj+s, p}^{(q)}$$

**Average Pooling**:
$$h_{ijp}^{(q+1)} = \frac{1}{P^2} \sum_{r,s \in [1,P]} h_{Si+r, Sj+s, p}^{(q)}$$

**Properties**:
- Operates on each feature map independently
- **Does not change depth**
- Typical: $2 \times 2$ pooling with stride 2
- Reduces each dimension by factor of 2

**Benefits**:
1. **Translation invariance**: Small shifts don't affect output much
2. **Receptive field increase**: Each neuron sees larger input region
3. **Dimensionality reduction**: Fewer parameters, less overfitting
4. **Computational efficiency**: Smaller feature maps

**Modern Trend**: Many architectures replace pooling with strided convolutions

## 8.5 Complete CNN Architecture

### 8.5.1 Layer Types

**1. Convolutional Layer**:
- Learns spatial features
- Parameters: filters, stride, padding

**2. Activation Layer** (ReLU):
- Applied element-wise
- Usually follows convolution

**3. Pooling Layer**:
- Downsampling
- No learnable parameters

**4. Fully Connected Layer**:
- At end of network
- Flattens spatial dimensions
- Standard dense layer

### 8.5.2 Typical CNN Pattern

```
[INPUT] → [[CONV → RELU] × N → POOL?] × M → [FC → RELU] × K → FC
```

where:
- $N \geq 1$: convolutions before pooling
- $M \geq 0$: number of conv-pool blocks
- $K \geq 0$: number of FC hidden layers

**Example**: VGG-16
```
INPUT (224×224×3)
→ [CONV3-64 → RELU] × 2 → POOL
→ [CONV3-128 → RELU] × 2 → POOL
→ [CONV3-256 → RELU] × 3 → POOL
→ [CONV3-512 → RELU] × 3 → POOL
→ [CONV3-512 → RELU] × 3 → POOL
→ FC-4096 → RELU → DROPOUT
→ FC-4096 → RELU → DROPOUT
→ FC-1000 (classes)
```

### 8.5.3 Receptive Field

**Receptive Field**: Region of input that affects a particular output

**Growth with layers**:
- After one $3 \times 3$ conv: $3 \times 3$ receptive field
- After two: $5 \times 5$
- After three: $7 \times 7$
- After $2 \times 2$ pooling: doubles

**Formula** (for $N$ layers with $F \times F$ filters):
$$R_N = 1 + N(F-1)$$

**Design principle**: Deep narrow filters > shallow wide filters
- Same receptive field, fewer parameters
- More nonlinearity
- Example: $3$ layers of $3 \times 3$ vs $1$ layer of $7 \times 7$

## 8.6 CNN Architectures (Case Studies)

### 8.6.1 LeNet-5 (1998)

**First successful CNN** for digit recognition (MNIST)

```
INPUT (32×32×1)
→ CONV5-6 → SIGMOID → POOL2
→ CONV5-16 → SIGMOID → POOL2
→ FC-120 → SIGMOID
→ FC-84 → SIGMOID
→ FC-10
```

**Key innovations**:
- Alternating conv-pool structure
- Used tanh/sigmoid (ReLU not yet discovered)

### 8.6.2 AlexNet (2012)

**ImageNet winner**, sparked deep learning revolution

```
INPUT (227×227×3)
→ CONV11-96, stride 4 → RELU → POOL3
→ CONV5-256 → RELU → POOL3
→ CONV3-384 → RELU
→ CONV3-384 → RELU
→ CONV3-256 → RELU → POOL3
→ FC-4096 → RELU → DROPOUT(0.5)
→ FC-4096 → RELU → DROPOUT(0.5)
→ FC-1000
```

**Innovations**:
- **ReLU activation**: 6× faster training
- **Dropout**: Reduced overfitting
- **Data augmentation**: Translations, flips
- **GPU training**: Parallelized across 2 GPUs

### 8.6.3 VGG (2014)

**Simple, deep architecture** with small filters

**VGG-16 configuration**:
- $3 \times 3$ convolutions throughout
- $2 \times 2$ max pooling
- 16 weight layers (13 conv + 3 FC)

**Key insight**: Stack of $3 \times 3$ convs = larger receptive field
- Two $3 \times 3$ = one $5 \times 5$
- Three $3 \times 3$ = one $7 \times 7$
- But fewer parameters and more nonlinearity

### 8.6.4 ResNet (2015)

**Residual connections** enable very deep networks (50-152 layers)

**Skip/Residual Connection**:
$$\mathbf{h}^{(\ell+1)} = \mathbf{h}^{(\ell)} + F(\mathbf{h}^{(\ell)})$$

where $F$ is 2-3 conv layers.

**Why it works**:
- **Gradient flow**: Gradients flow directly through skip connections
- **Identity mapping**: Easy to learn identity if needed
- **Solves vanishing gradient** in very deep networks

**Residual Block**:
```
Input
  |━━━━━━━━━━┐
  |           |
CONV3 → BN → RELU
  |
CONV3 → BN
  |           |
  +━━━━━━━━━━┘
  |
RELU → Output
```

**Impact**: Won ImageNet 2015, enabled 1000+ layer networks

## 8.7 Recurrent Neural Networks (RNNs)

RNNs are designed for **sequential data** with temporal dependencies.

### 8.7.1 Motivation

**Sequential Data**:
- Text: words in sentence
- Speech: audio samples over time
- Time-series: stock prices, sensor data
- Video: frames over time

**Challenges**:
- Variable length sequences
- Long-range dependencies
- Order matters

**Feedforward networks fail**:
- Fixed input size
- No memory of previous inputs
- Independent predictions

### 8.7.2 RNN Architecture

**Key idea**: Maintain **hidden state** that captures information from previous time steps

**Recurrence relation**:
$$\begin{align}
\mathbf{h}_t &= \tanh(\mathbf{W}_h \mathbf{h}_{t-1} + \mathbf{W}_x \mathbf{x}_t + \mathbf{b}_h) \\
\mathbf{o}_t &= \mathbf{W}_o \mathbf{h}_t + \mathbf{b}_o
\end{align}$$

where:
- $\mathbf{x}_t$: input at time $t$
- $\mathbf{h}_t$: hidden state at time $t$
- $\mathbf{o}_t$: output at time $t$
- $\mathbf{W}_x, \mathbf{W}_h, \mathbf{W}_o$: weight matrices (shared across time)

**Parameter sharing**: Same $\mathbf{W}$ applied at all time steps

### 8.7.3 RNN Forward Pass

```
Algorithm: RNN-FORWARD(sequence x_{1:T}, weights W)

begin
    h_0 ← 0  // Initialize hidden state
    
    for t = 1 to T do
        // Update hidden state
        a_t ← W_h h_{t-1} + W_x x_t + b_h
        h_t ← tanh(a_t)
        
        // Compute output
        o_t ← W_o h_t + b_o
        
        // Store for backprop
        store h_t, a_t, o_t
    
    return {o_1, ..., o_T}
end
```

**Complexity**: $O(T \cdot d_h^2)$ for sequence length $T$, hidden size $d_h$

## 8.8 Backpropagation Through Time (BPTT)

**Challenge**: RNN has temporal dependencies - must backpropagate through time

### 8.8.1 Loss Function

For sequence-to-sequence tasks:
$$\mathcal{L} = \sum_{t=1}^{T} \mathcal{L}_t(\mathbf{o}_t, \mathbf{y}_t)$$

### 8.8.2 BPTT Algorithm

```
Algorithm: BPTT(sequence x_{1:T}, targets y_{1:T}, weights W)

begin
    // Forward pass
    {o_1, ..., o_T}, {h_1, ..., h_T} ← RNN-FORWARD(x_{1:T}, W)
    
    // Compute total loss
    L ← ∑_t Loss(o_t, y_t)
    
    // Initialize gradient for last time step
    ∂L/∂h_T ← W_o^T (∂L/∂o_T)
    
    // Backward pass through time
    for t = T down to 1 do
        // Gradient w.r.t. output weights
        ∂L/∂W_o ← ∂L/∂W_o + (∂L/∂o_t) h_t^T
        ∂L/∂b_o ← ∂L/∂b_o + ∂L/∂o_t
        
        // Gradient w.r.t. hidden state (from output)
        ∂L/∂h_t ← W_o^T (∂L/∂o_t)
        
        // Add gradient from next time step
        if t < T then
            ∂L/∂h_t ← ∂L/∂h_t + W_h^T (∂L/∂a_{t+1})
        
        // Gradient through tanh
        ∂L/∂a_t ← ∂L/∂h_t ⊙ (1 - h_t^2)
        
        // Gradient w.r.t. weights
        ∂L/∂W_h ← ∂L/∂W_h + (∂L/∂a_t) h_{t-1}^T
        ∂L/∂W_x ← ∂L/∂W_x + (∂L/∂a_t) x_t^T
        ∂L/∂b_h ← ∂L/∂b_h + ∂L/∂a_t
    
    return {∂L/∂W_h, ∂L/∂W_x, ∂L/∂W_o, ∂L/∂b_h, ∂L/∂b_o}
end
```

### 8.8.3 Vanishing/Exploding Gradients

**Problem**: Gradients multiply through many time steps

$$\frac{\partial \mathcal{L}}{\partial \mathbf{h}_t} = \frac{\partial \mathcal{L}}{\partial \mathbf{h}_T} \prod_{i=t+1}^{T} \frac{\partial \mathbf{h}_i}{\partial \mathbf{h}_{i-1}}$$

**Vanishing**: If gradient norm less than 1, gradients decay exponentially
- Long-range dependencies not learned
- Early time steps get tiny gradients

**Exploding**: If gradient norm greater than 1, gradients explode
- Training becomes unstable
- **Solution**: Gradient clipping

Clip gradient if norm exceeds threshold.

## 8.9 Long Short-Term Memory (LSTM)

**LSTM** (Hochreiter & Schmidhuber, 1997) solves vanishing gradient problem with **gating mechanisms**.

### 8.9.1 LSTM Cell

**Key components**:
1. **Cell state** ($\mathbf{c}_t$): Long-term memory
2. **Hidden state** ($\mathbf{h}_t$): Short-term memory (output)
3. **Three gates**: Control information flow

### 8.9.2 LSTM Gates

**Forget Gate** (what to remove from cell state):
$$\mathbf{f}_t = \sigma(\mathbf{W}_f [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_f)$$

**Input Gate** (what new information to add):
$$\mathbf{i}_t = \sigma(\mathbf{W}_i [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_i)$$
$$\tilde{\mathbf{c}}_t = \tanh(\mathbf{W}_c [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_c)$$

**Output Gate** (what to output from cell state):
$$\mathbf{o}_t = \sigma(\mathbf{W}_o [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_o)$$

### 8.9.3 LSTM Update Equations

**Update cell state**:
$$\mathbf{c}_t = \mathbf{f}_t \odot \mathbf{c}_{t-1} + \mathbf{i}_t \odot \tilde{\mathbf{c}}_t$$

**Update hidden state**:
$$\mathbf{h}_t = \mathbf{o}_t \odot \tanh(\mathbf{c}_t)$$

where $\odot$ is element-wise multiplication.

### 8.9.4 LSTM Algorithm

```
Algorithm: LSTM-FORWARD(x_t, h_{t-1}, c_{t-1}, W)

begin
    // Concatenate inputs
    z_t ← [h_{t-1}, x_t]
    
    // Forget gate
    f_t ← σ(W_f z_t + b_f)
    
    // Input gate
    i_t ← σ(W_i z_t + b_i)
    c̃_t ← tanh(W_c z_t + b_c)
    
    // Output gate
    o_t ← σ(W_o z_t + b_o)
    
    // Update cell state
    c_t ← f_t ⊙ c_{t-1} + i_t ⊙ c̃_t
    
    // Update hidden state
    h_t ← o_t ⊙ tanh(c_t)
    
    return h_t, c_t
end
```

### 8.9.5 Why LSTM Works

**Gradient flow**:
$$\frac{\partial \mathbf{c}_t}{\partial \mathbf{c}_{t-1}} = \mathbf{f}_t$$

**Key properties**:
1. **Additive updates**: $\mathbf{c}_t = \mathbf{f}_t \odot \mathbf{c}_{t-1} + ...$
   - Gradients flow directly without repeated multiplication
2. **Gating**: Gates learned to preserve/forget information
3. **Long-range dependencies**: Cell state carries information far

**Compared to vanilla RNN**:
- RNN: Gradients multiply through $\mathbf{W}_h$ at each step
- LSTM: Gradients flow through cell state with minimal decay

## 8.10 Gated Recurrent Unit (GRU)

**GRU** (Cho et al., 2014) is a simpler alternative to LSTM.

### 8.10.1 GRU Gates

**Reset Gate**:
$$\mathbf{r}_t = \sigma(\mathbf{W}_r [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_r)$$

**Update Gate**:
$$\mathbf{z}_t = \sigma(\mathbf{W}_z [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_z)$$

**Candidate Hidden State**:
$$\tilde{\mathbf{h}}_t = \tanh(\mathbf{W}_h [\mathbf{r}_t \odot \mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_h)$$

**Hidden State Update**:
$$\mathbf{h}_t = (1 - \mathbf{z}_t) \odot \mathbf{h}_{t-1} + \mathbf{z}_t \odot \tilde{\mathbf{h}}_t$$

### 8.10.2 LSTM vs GRU

| Feature | LSTM | GRU |
|---------|------|-----|
| Gates | 3 (forget, input, output) | 2 (reset, update) |
| States | 2 (cell, hidden) | 1 (hidden) |
| Parameters | More | Fewer |
| Performance | Slightly better | Comparable |
| Training speed | Slower | Faster |

**When to use**:
- **LSTM**: Default choice, large datasets
- **GRU**: Limited data, faster training needed

## 8.11 Bidirectional RNNs

**Problem**: Standard RNN only uses past context

**Solution**: Process sequence in both directions

### 8.11.1 Architecture

**Forward RNN**: $\overrightarrow{\mathbf{h}}_t = f(\overrightarrow{\mathbf{h}}_{t-1}, \mathbf{x}_t)$

**Backward RNN**: $\overleftarrow{\mathbf{h}}_t = f(\overleftarrow{\mathbf{h}}_{t+1}, \mathbf{x}_t)$

**Combined output**: $\mathbf{h}_t = [\overrightarrow{\mathbf{h}}_t; \overleftarrow{\mathbf{h}}_t]$

### 8.11.2 Applications

**Good for**:
- Named entity recognition
- Part-of-speech tagging
- Machine translation
- Any task where full sentence context helps

**Not suitable for**:
- Online/streaming tasks
- Real-time generation
- Causal modeling

## 8.12 Summary

### CNNs

**Key concepts**:
- Convolution with filters
- Padding (half, valid, full)
- Strides for downsampling
- Pooling for translation invariance
- Hierarchical feature learning

**Architectures**:
- LeNet-5: First successful CNN
- AlexNet: Deep learning breakthrough
- VGG: Simple, deep with 3×3 filters
- ResNet: Skip connections for very deep networks

### RNNs

**Key concepts**:
- Recurrent connections for sequences
- Parameter sharing across time
- BPTT for training
- Vanishing/exploding gradients

**Solutions**:
- LSTM: Gates + cell state
- GRU: Simpler gating
- Bidirectional: Use future context
- Gradient clipping: Prevent explosion

### Algorithm Comparison

| Algorithm | Use Case | Pros | Cons |
|-----------|----------|------|------|
| CNN | Images, spatial data | Parameter efficient, translation invariant | Fixed input size |
| Vanilla RNN | Short sequences | Simple | Vanishing gradients |
| LSTM | Long sequences | Handles long dependencies | More parameters, slower |
| GRU | Medium sequences | Faster than LSTM | Slightly worse performance |
| Bidirectional | Full context available | Uses future info | Cannot stream |

## 8.13 Implementation

For complete Python implementations, see:

[ch08_domain_architectures_implementation.ipynb](ch08_domain_architectures_implementation.ipynb)

The implementation notebook includes:

**CNNs**:
1. Convolution operation from scratch
2. Padding implementations
3. Pooling layers
4. Complete CNN for MNIST
5. VGG-style architecture
6. ResNet block implementation

**RNNs**:
1. Vanilla RNN from scratch
2. BPTT implementation
3. LSTM cell
4. GRU cell
5. Bidirectional RNN
6. Sentiment analysis with LSTM
7. Text generation
8. Time-series forecasting

## Further Reading

### Textbooks

- Aggarwal, C. C. (2021). *Artificial Intelligence: A Textbook*. Springer. [Chapter 8]
- Goodfellow, I., Bengio, Y., & Courville, A. (2016). *Deep Learning*. MIT Press. [Chapters 9-10]

### Classic Papers

**CNNs**:
- LeCun, Y., et al. (1998). Gradient-based learning applied to document recognition. *Proceedings of the IEEE*.
- Krizhevsky, A., Sutskever, I., & Hinton, G. E. (2012). ImageNet classification with deep convolutional neural networks. *NIPS*.
- Simonyan, K., & Zisserman, A. (2015). Very deep convolutional networks for large-scale image recognition. *ICLR*.
- He, K., et al. (2016). Deep residual learning for image recognition. *CVPR*.

**RNNs**:
- Hochreiter, S., & Schmidhuber, J. (1997). Long short-term memory. *Neural Computation*.
- Cho, K., et al. (2014). Learning phrase representations using RNN encoder-decoder. *EMNLP*.
- Graves, A., et al. (2013). Speech recognition with deep recurrent neural networks. *ICASSP*.